# M1 walkthrough: realistic-looking id swaps and three cheap baselines
Plumbing only: default parameters, nothing tuned (DEVELOPMENT.md §6 M1). The corruption is ours, so the question here is whether everything runs and explains itself, not whether ArbiterKB wins.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.environ['ARBITER_DEBUG'] = '1'
import numpy as np
from arbiter_kb.sources.synthetic import SyntheticSource
from arbiter_kb.sources.realistic import OverlapSwapSource, NoisySource
from arbiter_kb.systems.arbiter_kb import ArbiterKBSystem
from arbiter_kb.systems.baselines import TrackerSystem, GuardedBound
from arbiter_kb.evaluate import run
from arbiter_kb.debug import explain
from arbiter_kb.params import Params
params = Params.load('../params.json')

In [ ]:
# lane 'mix': each episode draws a sideways offset; small offsets make the two containers overlap
base = SyntheticSource(lane_offset='mix', swap_steps='mix')
[(ep, base.settings(ep)) for ep in base.episodes('train', 8)]

In [ ]:
# id_swap_overlap: ids may swap only where the two tracks come within d_ov = 3 cm; the label jumps < d_ov
bad = OverlapSwapSource(base, p_swap=1.0, d_ov=params.baseline.d_ov)
ep = next(e for e in base.episodes('train', 40) if list(bad.frames(e)) and bad.log[e]['corrupted'])
base.settings(ep), bad.log[ep]

In [ ]:
# Six systems on that episode; correctness is physical (the label nearest the true container)
systems = [TrackerSystem('frozen', params), TrackerSystem('bound', params), GuardedBound('haircut', params),
           GuardedBound('overlap_haircut', params), GuardedBound('jump', params), ArbiterKBSystem(params)]
rows, traces, _ = run(systems, bad, [ep], params, 'id_swap_overlap')
[(r['system'], r['pred'], r['truth'], r['p_max'], r['correct']) for r in rows]

In [ ]:
# R6 explained: the near_pass evidence with min_3d, and the swap mixing weight it produced
cube = base.ground_truth(ep).targets[0]
print(explain(traces[('arbiterkb', ep)], cube))

In [ ]:
# Baselines explain too: one-hot answer plus the doubt trigger
print(explain(traces[('overlap_haircut', ep)], cube))

**Reading the table.** OverlapHaircut is Bayes-optimal under id_swap_overlap because we wrote both: ArbiterKB − OverlapHaircut ≈ 0 here is expected and says nothing about real trackers. That is why phase G measures a real tracker's swaps before M2 tunes anything. Read silent failure together with coverage (share answered with p_max ≥ 0.8): a 50/50 haircut scores 0 silent failures by never being sure. AURC summarises the whole risk–coverage curve. JumpDetector equals Bound here (jumps < j_max) and collapses under position noise.

In [ ]:
import pandas as pd, glob
pd.read_csv('../results/latest/m1_table/summary.csv')[['condition', 'system', 'acc', 'silent', 'coverage', 'aurc', 'brier', 'silent_on_corrupted']]

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob('../figures/*_test_p0.5.png')):
    print(f); display(Image(f))